# 01 — Forecasting LSTM Baseline
### LSTM binary forecaster — Baseline 1
> Run 00_Data_Preparation.ipynb first.
>
> Predicts: will a breakdown happen within the next 5 pedal presses?


In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import f1_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device: {DEVICE}')


In [ ]:
with open('prepared_forecasting.pkl','rb') as f: d = pickle.load(f)

X_train = torch.FloatTensor(d['X_fc_train_seq']).to(DEVICE)
X_test  = torch.FloatTensor(d['X_fc_test_seq']).to(DEVICE)
y_train = torch.LongTensor(d['y_fc_train_bin_seq']).to(DEVICE)
y_test  = torch.LongTensor(d['y_fc_test_bin_seq']).to(DEVICE)

NUM_FEATURES = d['num_features']
TIME_STEPS   = d['TIME_STEPS']

# CLASS WEIGHTS (binary imbalance)
class_counts  = np.bincount(d['y_fc_train_bin_seq'])
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)
weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
print(f'✅ Data loaded. Features: {NUM_FEATURES}, TIME_STEPS: {TIME_STEPS}')
print(f'   Class counts  : {class_counts}')
print(f'   Class weights : {class_weights.round(3)}')

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)


#### LSTM Forecaster Architecture


In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout)
        self.bn   = nn.BatchNorm1d(hidden_size)
        self.fc1  = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.fc2  = nn.Linear(32, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.bn(out[:, -1, :])
        out = self.drop(self.relu(self.fc1(out)))
        return self.fc2(out)

model = LSTMForecaster(NUM_FEATURES, hidden_size=64, num_layers=2,
                       num_classes=2).to(DEVICE)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')


#### Train


In [ ]:
EPOCHS    = 150
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=8)

best_val_loss    = float('inf')
patience_counter = 0
PATIENCE         = 20
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    batch_losses = []
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        batch_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_test), y_test).item()
    train_loss = np.mean(batch_losses)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_forecast_lstm.pt')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}')
            break

    print(f'Epoch {epoch+1:3d} | Train: {train_loss:.4f} | Val: {val_loss:.4f}')

plt.figure(figsize=(8,4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.title('Forecasting LSTM — Loss Curves')
plt.legend()
plt.tight_layout()
plt.savefig('forecast_lstm_loss.png', dpi=150)
plt.show()


#### Evaluate


In [ ]:
model.load_state_dict(torch.load('best_forecast_lstm.pt'))
model.eval()
with torch.no_grad():
    y_pred = np.argmax(torch.softmax(model(X_test),dim=1).cpu().numpy(), axis=1)
    y_true = y_test.cpu().numpy()

acc  = accuracy_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred, average='weighted')
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)

print('='*50)
print('FORECASTING LSTM BASELINE RESULTS')
print('='*50)
print(f'Accuracy : {acc*100:.2f}%')
print(f'F1 Score : {f1:.4f}')
print(f'RMSE     : {rmse:.4f}')
print(f'MAE      : {mae:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=['Safe','Breakdown Coming']))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Safe','Breakdown Coming'],
            yticklabels=['Safe','Breakdown Coming'])
plt.title('Forecasting LSTM — Confusion Matrix')
plt.tight_layout()
plt.savefig('forecast_lstm_confusion.png', dpi=150)
plt.show()

lstm_results = {'model':'LSTM Forecaster','accuracy':acc,'f1':f1,'rmse':rmse,'mae':mae}
with open('forecast_lstm_results.pkl','wb') as f: pickle.dump(lstm_results, f)
print('✅ Results saved.')
